# 🎓 Orquestador Maestro — AU_UJI Dinámico (V2)

**TFM: Pronóstico del Éxito y del Abandono en los Títulos de Grado de la Universitat Jaume I**

| | |
|---|---|
| **Autora** | María José Morte Ruiz |
| **Institución** | UOC + Universitat Jaume I |
| **Email** | mjmorteruiz@uoc.edu · morte@uji.es |
| **Tutor UOC** | Raúl Parada |
| **Supervisora UJI** | Susana |
| **Versión** | AU_UJI Dinámico (V2) |

---

## 🎯 ¿Qué hace este notebook?

Este es el **orquestador maestro** que ejecuta en cadena los orquestadores
de cada fase del TFM, regenerando todos los HTMLs del proyecto.

**Está diseñado para que el tribunal o el profesor ejecute todas las fases
de un golpe y vea el progreso en tiempo real**, con la opción de **parar
o continuar tras cada fase**.

Al finalizar (o al parar), el notebook **genera automáticamente** el HTML
`docs/html/orquestador_resumen.html` con un resumen dinámico de la
ejecución y del estado actual del proyecto. Este HTML también se puede
regenerar manualmente desde `f0_actualizar_resumen.ipynb`.

## 🧭 Cómo se usa

1. Ejecutar las celdas 1–4 (configuración, verificación de orquestadores y **validación pre-fase**)
2. Ejecutar la celda 5 → aparece el **panel de control**
3. Pulsar **▶️ Comenzar Fase 1**
4. Tras cada fase aparecen dos botones:
   - **▶️ Continuar con Fase X+1** — sigue con la siguiente
   - **⏸️ Parar aquí** — detiene la ejecución y muestra resumen parcial
5. Al final aparecen dos botones:
   - **✅ Ver resumen final** — muestra resumen en pantalla y genera HTML
   - **🔄 Refrescar resumen HTML** — regenera el HTML sin reejecutar nada

## 📋 Requisitos

- Todos los orquestadores de fase en sus carpetas (`fX_m00_ejecucion.ipynb`)
- `src/utils/orquestador.py` con función `ejecutar_notebook()`
- `src/html/generar_resumen_proyecto.py` con función `generar_resumen_orquestador()`
- Entorno: `tfm_abandono` (con `nbconvert`, `ipywidgets`)
- Paquete `src/validacion/` (validador profundo N1-N5 que se ejecuta antes de F1)

## ⚠️ Advertencia de tiempo

La ejecución completa puede tardar **varias horas**. La Fase 5 (modelado)
y la Fase 6 (SHAP) son las más lentas. Se recomienda:

- Lanzar fase a fase usando los botones de pausa
- Dejar el ordenador conectado a la red eléctrica
- No cerrar el navegador con Jupyter abierto durante la ejecución

## 🌐 Aplicación Web (Fase 7)

La Fase 7 **no se ejecuta** desde aquí. Se trata de una aplicación web
desplegada en Streamlit Cloud. El orquestador mostrará un cartel
informativo con los enlaces relevantes.

- **App V2 (esta versión):** https://tfm-abandono-dinamico.streamlit.app
- **Repositorio:** https://github.com/mortemj/AU_UJI_v2
- **Documentación:** https://mortemj.github.io/AU_UJI_v2/

## ➡️ Salida

- Regenera todos los HTMLs del proyecto en `docs/html/`
- Genera `docs/html/orquestador_resumen.html` (resumen dinámico)
- Tras finalizar, abrir `docs/html/index.html` o visitar
  https://mortemj.github.io/AU_UJI_v2/


In [ ]:
# ============================================================================
# CELDA 1 — CONFIGURACIÓN DE RUTAS (ROOT robusto)
# ============================================================================
import sys
from pathlib import Path

def _encontrar_root(start: Path) -> Path:
    """Sube por los padres hasta encontrar la carpeta src/."""
    for parent in [start] + list(start.parents):
        if (parent / 'src').is_dir():
            return parent
    raise FileNotFoundError(f'No se encontró src/ subiendo desde {start}')

ROOT = _encontrar_root(Path.cwd())
sys.path.insert(0, str(ROOT))

DIR_NOTEBOOKS = ROOT / 'notebooks'

print(f'ROOT:          {ROOT}')
print(f'DIR_NOTEBOOKS: {DIR_NOTEBOOKS}')


In [ ]:
# ============================================================================
# CELDA 2 — IMPORTS
# ============================================================================
import time
from datetime import datetime
from IPython.display import display, HTML, clear_output
import ipywidgets as widgets

from src.utils.orquestador import ejecutar_notebook
from src.html.generar_resumen_proyecto import generar_resumen_orquestador

print('✓ Imports correctos')
print(f'✓ ipywidgets {widgets.__version__}')


In [ ]:
# ============================================================================
# CELDA 3 — LISTA DE FASES Y VERIFICACIÓN
# ============================================================================
# Cada entrada es un dict con:
#   - id        : identificador corto (para botones y barras)
#   - nombre    : nombre completo (cabecera del bloque)
#   - emoji     : icono visible
#   - carpeta   : subcarpeta dentro de notebooks/
#   - notebook  : fichero orquestador de la fase (None si no aplica, p.e. F7)
#   - es_app    : True si es una fase tipo aplicación (no se ejecuta nada)
#
# Para excluir/incluir/reordenar fases, edita esta lista.
# ============================================================================

FASES = [
    {'id': 'f1', 'nombre': 'Fase 1 — Transformación',       'emoji': '🧹',
     'carpeta': 'fase1_transformacion', 'notebook': 'f1_m00_ejecucion.ipynb', 'es_app': False},
    {'id': 'f2', 'nombre': 'Fase 2 — EDA Datos Originales', 'emoji': '🔍',
     'carpeta': 'fase2_eda',            'notebook': 'f2_m00_ejecucion.ipynb', 'es_app': False},
    {'id': 'f3', 'nombre': 'Fase 3 — Features',             'emoji': '🧪',
     'carpeta': 'fase3_features',       'notebook': 'f3_m00_ejecucion.ipynb', 'es_app': False},
    {'id': 'fa', 'nombre': 'Pre-Modelado AutoML',           'emoji': '🤖',
     'carpeta': 'fase_automl',          'notebook': 'fautoml_m00_ejecucion.ipynb', 'es_app': False},
    {'id': 'f4', 'nombre': 'Fase 4 — EDA Final',            'emoji': '📊',
     'carpeta': 'fase4_eda',            'notebook': 'f4_m00_ejecucion.ipynb', 'es_app': False},
    {'id': 'f5', 'nombre': 'Fase 5 — Modelado',             'emoji': '🚀',
     'carpeta': 'fase5_modelado',       'notebook': 'f5_m00_ejecucion.ipynb', 'es_app': False},
    {'id': 'f6', 'nombre': 'Fase 6 — Evaluación',           'emoji': '⚖️',
     'carpeta': 'fase6_evaluacion',     'notebook': 'f6_m00_ejecucion.ipynb', 'es_app': False},
    {'id': 'f7', 'nombre': 'Fase 7 — Aplicación Web',       'emoji': '🌐',
     'carpeta': 'fase7_app',            'notebook': None,                     'es_app': True},
]

TIMEOUT = 3600  # segundos por celda en nbconvert

# --- Verificación ---
print(f'{len(FASES)} fases configuradas:\n')
for fase in FASES:
    if fase['es_app']:
        marca = '🌐'
        info = '(no ejecuta nada — cartel informativo)'
    else:
        ruta = DIR_NOTEBOOKS / fase['carpeta'] / fase['notebook']
        marca = '✅' if ruta.exists() else '❌'
        info = str(ruta.relative_to(ROOT)) if ruta.exists() else f'NO ENCONTRADO: {ruta}'
    print(f'  {marca}  {fase["emoji"]} {fase["nombre"]:35s}  {info}')


In [ ]:
# ============================================================================
# CELDA 3.5 — VALIDACIÓN PRE-FASE (N1-N5)
# ============================================================================
# Antes de lanzar las fases del proyecto, ejecutamos la validación profunda
# de los 2 Excel originales (paquete src.validacion). Si fallan validaciones
# bloqueantes (N1-N3) NO se permite continuar.
#
# Esta celda implementa la integración profesional descrita en TAREA C
# (`f0_validar_excel.ipynb` se puede ejecutar también de forma standalone).
#
# Tiempo aproximado: 2-3 minutos (N5 carga columnas grandes para validar
# cruces de IDs entre hojas).
# ============================================================================

print('=' * 60)
print('VALIDACIÓN PRE-FASE (N1-N5)')
print('=' * 60)
print('Antes de lanzar las fases, validamos que los 2 Excel originales')
print('cumplen el contrato esperado. Tiempo: ~2-3 minutos.')
print()

from src.validacion.validador_excel import ejecutar_validacion_completa
from src.validacion.generar_html_validacion import generar_html_validacion

# Ejecutar las 5 validaciones (verbose=True para ver progreso)
_resultado_validacion = ejecutar_validacion_completa(verbose=True)

print()
print(_resultado_validacion['resumen_corto'])
print()

# Generar SIEMPRE el HTML (útil tanto si pasa como si falla)
_ruta_html_validacion = generar_html_validacion(_resultado_validacion)
print(f'  📄 Informe HTML: {_ruta_html_validacion.relative_to(ROOT)}')
print()

# --- Decisión: ¿continuar o abortar? ---
if _resultado_validacion['bloqueante_fallido']:
    print('=' * 60)
    print('❌ VALIDACIÓN FALLIDA — fases NO ejecutadas')
    print('=' * 60)
    print('Uno o más niveles bloqueantes (N1-N3) no han pasado.')
    print('Revisar el informe HTML y corregir antes de re-ejecutar.')
    raise RuntimeError(
        'Validación pre-fase FALLIDA: bloqueantes (N1-N3) no superados. '
        f'Ver detalles en {_ruta_html_validacion.relative_to(ROOT)}'
    )
elif _resultado_validacion['todos_ok']:
    print('✅ Validación pre-fase COMPLETADA — los 5 niveles OK.')
    print('   Se puede continuar con las fases F1-F6.')
else:
    print('⚠️  Validación pre-fase con AVISOS en N4/N5 (no bloquea).')
    print('   Las fases pueden continuar; revisar el HTML para los detalles.')

print()
print('=' * 60)


In [ ]:
# ============================================================================
# CELDA 4 — ESTILOS Y HELPERS DE UI
# ============================================================================
# Define la paleta del orquestador, generadores de tarjetas HTML, y
# funciones para renderizar progreso. Todo el rendering de la celda 5
# delega aquí para mantener la celda principal corta.
# ============================================================================

# --- Paleta (alineada con config_app.py de la app V2) ---
COLOR_PRIMARIO  = '#1e4d8c'   # azul institucional UJI
COLOR_EXITO     = '#10b981'   # verde
COLOR_ERROR     = '#dc2626'   # rojo
COLOR_AVISO     = '#f59e0b'   # ámbar
COLOR_FONDO     = '#f7fafc'
COLOR_BORDE     = '#e2e8f0'
COLOR_TEXTO     = '#2d3748'
COLOR_TEXTO_SUAVE = '#718096'

ICONO_ESPERA    = '⏳'
ICONO_OK        = '✅'
ICONO_ERROR     = '❌'
ICONO_RUNNING   = '▶️'
ICONO_APP       = '🌐'
ICONO_PENDIENTE = '⚪'

# --- Estado global del orquestador (mutable durante la ejecución) ---
ESTADO = {
    'fase_actual_idx': -1,
    'resultados': [],          # lista de dicts {id, nombre, ok, info, minutos}
    'parado': False,
    't_inicio_global': None,
}


def _tarjeta_resumen_inicial():
    """Cabecera con título grande y fases que se van a ejecutar."""
    filas = []
    for fase in FASES:
        filas.append(
            f'<tr>'
            f'<td style="padding:6px 10px;">{ICONO_PENDIENTE}</td>'
            f'<td style="padding:6px 10px;">{fase["emoji"]} {fase["nombre"]}</td>'
            f'<td style="padding:6px 10px; color:{COLOR_TEXTO_SUAVE};">pendiente</td>'
            f'</tr>'
        )
    tabla = '<table style="width:100%; border-collapse:collapse;">' + ''.join(filas) + '</table>'
    return (
        f'<div style="border:2px solid {COLOR_PRIMARIO}; border-radius:10px;'
        f' padding:18px; background:{COLOR_FONDO};">'
        f'<h2 style="margin:0 0 10px 0; color:{COLOR_PRIMARIO};">'
        f'🎓 Orquestador Maestro — AU_UJI Dinámico (V2)</h2>'
        f'<p style="margin:0 0 14px 0; color:{COLOR_TEXTO_SUAVE};">'
        f'Pulsa <strong>▶️ Comenzar Fase 1</strong> para iniciar la ejecución. '
        f'Tras cada fase podrás <strong>continuar</strong> o <strong>parar</strong>.'
        f'</p>'
        f'{tabla}'
        f'</div>'
    )


def _tarjeta_estado_global(idx_actual: int, fase_estado: str):
    """Tabla con el estado de cada fase: ⏳/✅/❌/⚪."""
    filas = []
    for i, fase in enumerate(FASES):
        if i < idx_actual:
            r = next((x for x in ESTADO['resultados'] if x['id'] == fase['id']), None)
            if r is None:
                icono, texto, color = ICONO_PENDIENTE, 'pendiente', COLOR_TEXTO_SUAVE
            elif r['ok']:
                icono, texto, color = ICONO_OK, r['info'], COLOR_EXITO
            else:
                icono, texto, color = ICONO_ERROR, r['info'], COLOR_ERROR
        elif i == idx_actual:
            icono = ICONO_RUNNING if fase_estado == 'corriendo' else ICONO_ESPERA
            texto = fase_estado
            color = COLOR_PRIMARIO
        else:
            icono, texto, color = ICONO_PENDIENTE, 'pendiente', COLOR_TEXTO_SUAVE
        filas.append(
            f'<tr>'
            f'<td style="padding:6px 10px; width:32px;">{icono}</td>'
            f'<td style="padding:6px 10px;">{fase["emoji"]} {fase["nombre"]}</td>'
            f'<td style="padding:6px 10px; color:{color};">{texto}</td>'
            f'</tr>'
        )
    tabla = '<table style="width:100%; border-collapse:collapse;">' + ''.join(filas) + '</table>'
    return (
        f'<div style="border:1px solid {COLOR_BORDE}; border-radius:10px;'
        f' padding:14px; background:white; margin-top:10px;">'
        f'<h4 style="margin:0 0 8px 0; color:{COLOR_PRIMARIO};">📋 Estado global</h4>'
        f'{tabla}</div>'
    )


def _tarjeta_app_f7():
    """Cartel informativo para Fase 7 (no ejecutable)."""
    return (
        f'<div style="border:2px solid {COLOR_PRIMARIO}; border-radius:10px;'
        f' padding:18px; background:{COLOR_FONDO}; margin-top:14px;">'
        f'<h3 style="margin:0 0 10px 0; color:{COLOR_PRIMARIO};">'
        f'🌐 Fase 7 — Aplicación Web (AU_UJI Dinámico)</h3>'
        f'<p style="margin:0 0 10px 0; color:{COLOR_TEXTO};">'
        f'La aplicación <strong>no requiere ejecución local</strong>. '
        f'Está desplegada en Streamlit Cloud.</p>'
        f'<ul style="margin:0; line-height:1.8;">'
        f'<li>👉 <strong>App V2 (esta versión):</strong> '
        f'<a href="https://tfm-abandono-dinamico.streamlit.app" target="_blank">'
        f'https://tfm-abandono-dinamico.streamlit.app</a></li>'
        f'<li>📦 <strong>Repositorio:</strong> '
        f'<a href="https://github.com/mortemj/AU_UJI_v2" target="_blank">'
        f'https://github.com/mortemj/AU_UJI_v2</a></li>'
        f'<li>📖 <strong>Documentación:</strong> '
        f'<a href="https://mortemj.github.io/AU_UJI_v2/" target="_blank">'
        f'https://mortemj.github.io/AU_UJI_v2/</a></li>'
        f'<li>🔁 <strong>Versión anterior estable (V1):</strong> '
        f'<a href="https://tfm-abandono.streamlit.app" target="_blank">'
        f'https://tfm-abandono.streamlit.app</a></li>'
        f'</ul></div>'
    )


def _tarjeta_resumen_final():
    """Resumen final con totales y tabla detallada."""
    n_total = len(ESTADO['resultados'])
    n_ok    = sum(1 for r in ESTADO['resultados'] if r['ok'])
    n_err   = n_total - n_ok
    t_total_min = (time.time() - ESTADO['t_inicio_global']) / 60 if ESTADO['t_inicio_global'] else 0

    color_global = COLOR_EXITO if n_err == 0 and n_total > 0 else (
                   COLOR_AVISO if n_ok > 0 else COLOR_ERROR)
    titulo = ('🎉 Ejecución completada' if not ESTADO['parado']
              else '⏸️ Ejecución detenida por el usuario')

    filas = []
    for r in ESTADO['resultados']:
        icono = ICONO_OK if r['ok'] else ICONO_ERROR
        color = COLOR_EXITO if r['ok'] else COLOR_ERROR
        filas.append(
            f'<tr>'
            f'<td style="padding:6px 10px;">{icono}</td>'
            f'<td style="padding:6px 10px;">{r["nombre"]}</td>'
            f'<td style="padding:6px 10px; color:{color};">{r["info"]}</td>'
            f'</tr>'
        )
    tabla = '<table style="width:100%; border-collapse:collapse;">' + ''.join(filas) + '</table>'

    return (
        f'<div style="border:2px solid {color_global}; border-radius:10px;'
        f' padding:18px; background:white; margin-top:14px;">'
        f'<h2 style="margin:0 0 8px 0; color:{color_global};">{titulo}</h2>'
        f'<p style="margin:0 0 14px 0; color:{COLOR_TEXTO};">'
        f'<strong>{n_ok}/{n_total}</strong> fases completadas correctamente · '
        f'<strong>{n_err}</strong> con error · '
        f'Tiempo total: <strong>{t_total_min:.1f} min</strong></p>'
        f'{tabla}</div>'
    )


print('✓ Estilos y helpers definidos')


In [ ]:
# ============================================================================
# CELDA 5 — PANEL DE CONTROL (barra de progreso + pausas + HTML resumen)
# ============================================================================
# Renderiza el panel de control con tres widgets:
#   - Cabecera (ya generada en celda 4)
#   - Barra de progreso global (FloatProgress)
#   - Output que muestra la fase en curso, su barra propia y los botones
#     de continuar/parar tras cada fase.
#
# El profesor o el tribunal pulsa ▶️ Comenzar Fase 1 → arranca la ejecución.
# Tras cada fase aparecen 2 botones:
#   - ▶️ Continuar con Fase N+1
#   - ⏸️ Parar aquí
# Al final (terminada o parada) aparecen 2 botones:
#   - ✅ Ver resumen final (también genera el HTML resumen)
#   - 🔄 Refrescar resumen HTML (regenera sin reejecutar)
# ============================================================================

# --- Widgets ---
out_cabecera   = widgets.Output()
barra_global   = widgets.FloatProgress(
    value=0, min=0, max=len(FASES),
    description='Progreso:', bar_style='info',
    style={'description_width': '90px'},
    layout=widgets.Layout(width='100%')
)
out_estado     = widgets.Output()
out_fase_actual= widgets.Output()
out_botones    = widgets.Output()
out_resumen    = widgets.Output()
out_html_status= widgets.Output()

# Botón inicial
btn_comenzar = widgets.Button(
    description='▶️ Comenzar Fase 1',
    button_style='primary',
    layout=widgets.Layout(width='auto', height='44px')
)

# --- Mostrar cabecera + panel ---
with out_cabecera:
    display(HTML(_tarjeta_resumen_inicial()))

display(out_cabecera, btn_comenzar, barra_global, out_estado,
        out_fase_actual, out_botones, out_resumen, out_html_status)


def _refrescar_estado(idx_actual: int, texto_estado: str):
    out_estado.clear_output(wait=True)
    with out_estado:
        display(HTML(_tarjeta_estado_global(idx_actual, texto_estado)))


def _generar_html_resumen():
    """Llama al generador HTML y muestra el resultado en out_html_status."""
    out_html_status.clear_output(wait=True)
    try:
        ruta = generar_resumen_orquestador(
            resultados=ESTADO['resultados'],
            t_inicio_global=ESTADO['t_inicio_global'],
            parado=ESTADO['parado'],
            verbose=False,
        )
        with out_html_status:
            display(HTML(
                f'<div style="padding:14px; border:2px solid {COLOR_EXITO};'
                f' border-radius:10px; background:#f0fff4; margin-top:10px;">'
                f'<strong style="color:{COLOR_EXITO};">✅ HTML resumen generado</strong><br>'
                f'<code style="font-size:12px;">{ruta}</code><br>'
                f'<span style="color:{COLOR_TEXTO_SUAVE}; font-size:12px;">'
                f'Abre el fichero en el navegador o visita la web del proyecto.</span></div>'
            ))
    except Exception as e:
        with out_html_status:
            display(HTML(
                f'<div style="padding:14px; border:2px solid {COLOR_ERROR};'
                f' border-radius:10px; background:white; margin-top:10px;">'
                f'<strong style="color:{COLOR_ERROR};">❌ Error generando HTML resumen</strong><br>'
                f'<code style="font-size:12px;">{e.__class__.__name__}: {e}</code></div>'
            ))


def _ejecutar_fase(fase: dict, idx: int):
    """Ejecuta una fase concreta, actualiza estado y barra."""
    ESTADO['fase_actual_idx'] = idx
    out_fase_actual.clear_output(wait=True)

    # ---- Caso especial: F7 (app, no se ejecuta) ----
    if fase['es_app']:
        with out_fase_actual:
            display(HTML(_tarjeta_app_f7()))
        ESTADO['resultados'].append({
            'id': fase['id'], 'nombre': fase['nombre'],
            'ok': True, 'info': 'cartel informativo (no ejecutable)',
            'minutos': 0,
        })
        barra_global.value = idx + 1
        _refrescar_estado(idx, 'app — cartel mostrado')
        return True

    # ---- Caso normal: ejecutar el orquestador de fase ----
    ruta_nb = DIR_NOTEBOOKS / fase['carpeta'] / fase['notebook']
    if not ruta_nb.exists():
        with out_fase_actual:
            display(HTML(
                f'<div style="padding:14px; border:2px solid {COLOR_ERROR};'
                f' border-radius:10px; background:white;">'
                f'<strong style="color:{COLOR_ERROR};">❌ Orquestador no encontrado</strong><br>'
                f'<code>{ruta_nb}</code></div>'
            ))
        ESTADO['resultados'].append({
            'id': fase['id'], 'nombre': fase['nombre'],
            'ok': False, 'info': 'orquestador no encontrado',
            'minutos': 0,
        })
        barra_global.value = idx + 1
        _refrescar_estado(idx, 'no encontrado')
        return False

    # Mostrar tarjeta "fase en curso"
    with out_fase_actual:
        display(HTML(
            f'<div style="padding:14px; border:2px solid {COLOR_PRIMARIO};'
            f' border-radius:10px; background:white;">'
            f'<strong style="color:{COLOR_PRIMARIO}; font-size:16px;">'
            f'{ICONO_RUNNING} {fase["emoji"]} {fase["nombre"]}</strong><br>'
            f'<span style="color:{COLOR_TEXTO_SUAVE};">'
            f'Ejecutando <code>{fase["notebook"]}</code> con nbconvert…<br>'
            f'(esto puede tardar varios minutos según la fase)</span></div>'
        ))

    _refrescar_estado(idx, 'corriendo…')

    # Ejecutar
    t_ini = time.time()
    ok = ejecutar_notebook(ruta_nb, timeout=TIMEOUT)
    t_min = (time.time() - t_ini) / 60

    ESTADO['resultados'].append({
        'id': fase['id'], 'nombre': fase['nombre'],
        'ok': ok, 'info': f'{t_min:.1f} min' if ok else f'ERROR ({t_min:.1f} min)',
        'minutos': t_min,
    })

    # Actualizar tarjeta
    color = COLOR_EXITO if ok else COLOR_ERROR
    icono = ICONO_OK if ok else ICONO_ERROR
    out_fase_actual.clear_output(wait=True)
    with out_fase_actual:
        display(HTML(
            f'<div style="padding:14px; border:2px solid {color};'
            f' border-radius:10px; background:white;">'
            f'<strong style="color:{color}; font-size:16px;">'
            f'{icono} {fase["emoji"]} {fase["nombre"]}</strong> '
            f'— completada en <strong>{t_min:.1f} min</strong></div>'
        ))

    barra_global.value = idx + 1
    _refrescar_estado(idx, f'{t_min:.1f} min')
    return ok


def _mostrar_botones_continuar(idx_terminado: int):
    """Muestra los botones tras una fase. El último idx muestra finalizar + refrescar."""
    out_botones.clear_output(wait=True)

    if idx_terminado >= len(FASES) - 1:
        # Última fase: dos botones (finalizar + refrescar HTML)
        btn_fin = widgets.Button(
            description='✅ Ver resumen final',
            button_style='success',
            layout=widgets.Layout(width='auto', height='40px')
        )
        btn_refresh = widgets.Button(
            description='🔄 Refrescar resumen HTML',
            button_style='info',
            layout=widgets.Layout(width='auto', height='40px')
        )

        def _cb_fin(_btn):
            out_botones.clear_output()
            out_resumen.clear_output(wait=True)
            with out_resumen:
                display(HTML(_tarjeta_resumen_final()))
            _generar_html_resumen()

        def _cb_refresh(_btn):
            _generar_html_resumen()

        btn_fin.on_click(_cb_fin)
        btn_refresh.on_click(_cb_refresh)
        with out_botones:
            display(widgets.HBox([btn_fin, btn_refresh]))
        return

    fase_siguiente = FASES[idx_terminado + 1]
    btn_seguir = widgets.Button(
        description=f'▶️ Continuar con {fase_siguiente["nombre"]}',
        button_style='primary',
        layout=widgets.Layout(width='auto', height='40px')
    )
    btn_parar = widgets.Button(
        description='⏸️ Parar aquí',
        button_style='warning',
        layout=widgets.Layout(width='auto', height='40px')
    )

    def _cb_seguir(_btn):
        out_botones.clear_output()
        _ejecutar_y_pausar(idx_terminado + 1)

    def _cb_parar(_btn):
        ESTADO['parado'] = True
        out_botones.clear_output()
        out_resumen.clear_output(wait=True)
        with out_resumen:
            display(HTML(_tarjeta_resumen_final()))
        _generar_html_resumen()

    btn_seguir.on_click(_cb_seguir)
    btn_parar.on_click(_cb_parar)
    with out_botones:
        display(widgets.HBox([btn_seguir, btn_parar]))


def _ejecutar_y_pausar(idx: int):
    """Ejecuta la fase idx y muestra los botones de continuar/parar."""
    _ejecutar_fase(FASES[idx], idx)
    _mostrar_botones_continuar(idx)


def _on_comenzar(_btn):
    btn_comenzar.disabled = True
    btn_comenzar.description = '▶️ Ejecutando…'
    ESTADO['t_inicio_global'] = time.time()
    _ejecutar_y_pausar(0)


btn_comenzar.on_click(_on_comenzar)


In [ ]:
# ============================================================================
# CELDA 6 — NOTAS PARA EL TRIBUNAL
# ============================================================================

print('=' * 60)
print('NOTAS PARA EL TRIBUNAL / PROFESOR')
print('=' * 60)
print("""
1. El orquestador maestro NO recalcula los modelos pesados.
   Carga los .pkl ya entrenados desde data/05_modelado/models/
   y data/06_evaluacion/. Para reentrenar todo desde cero, hay que
   ejecutar Fase 5 con flag de reentrenamiento (ver f5_m00_ejecucion).

2. Si una fase falla, la ejecución NO se interrumpe automáticamente:
   se marca con ❌ y aparecen los botones para que el tribunal decida
   continuar o parar.

3. Tiempos orientativos por fase (varían según equipo):
     Fase 1 — Transformación      ~5-10 min
     Fase 2 — EDA Originales      ~10-15 min
     Fase 3 — Features            ~5-10 min
     Pre-Modelado AutoML          ~10-15 min (carga desde parquets)
     Fase 4 — EDA Final           ~5-10 min
     Fase 5 — Modelado            ~15-30 min (carga desde .pkl)
     Fase 6 — Evaluación          ~15-25 min (SHAP carga desde .pkl)
     Fase 7 — Aplicación Web      INSTANTÁNEO (cartel informativo)

4. La aplicación web (Fase 7) NO se ejecuta desde aquí. Tras finalizar
   el orquestador, el tribunal puede acceder directamente a:
   https://tfm-abandono-dinamico.streamlit.app
""")
